In [88]:
#  Import Libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

In [89]:
#load dataset
df = pd.read_csv("train.csv")

In [90]:

#  Handle Missing Values

# Numerical
df['LoanAmount'] = df['LoanAmount'].fillna(df['LoanAmount'].mean())
df['Loan_Amount_Term'] = df['Loan_Amount_Term'].fillna(df['Loan_Amount_Term'].mean())
df['Credit_History'] = df['Credit_History'].fillna(df['Credit_History'].mean())

# Categorical
df['Gender'] = df['Gender'].fillna(df['Gender'].mode()[0])
df['Married'] = df['Married'].fillna(df['Married'].mode()[0])
df['Dependents'] = df['Dependents'].fillna(df['Dependents'].mode()[0])
df['Self_Employed'] = df['Self_Employed'].fillna(df['Self_Employed'].mode()[0])

In [91]:
# Feature Engineering

df['Total_income'] = df['ApplicantIncome'] + df['CoapplicantIncome']
df['ApplicantIncomeLog'] = np.log(df['ApplicantIncome'])
df['LoanAmountLog'] = np.log(df['LoanAmount'])
df['Loan_Amount_Term_Log'] = np.log(df['Loan_Amount_Term'])
df['Total_Income_Log'] = np.log(df['Total_income'])
df['Income_Loan_Ratio'] = df['Total_Income_Log'] / df['LoanAmountLog']
df['Loan_Term_Ratio'] = df['LoanAmountLog'] / df['Loan_Amount_Term_Log']
df['Income_Credit'] = df['Total_Income_Log'] * df['Credit_History']

# Drop unnecessary columns
df.drop(columns=['ApplicantIncome','CoapplicantIncome',
                 'LoanAmount','Loan_Amount_Term',
                 'Total_income','Loan_ID'], inplace=True)

In [92]:
#  Encode Categorical Variables

df = pd.get_dummies(df, drop_first=True)

In [93]:

df['Loan_Status_Y'] = df['Loan_Status_Y'].astype(int)

X = df.drop(columns=['Loan_Status_Y'])
y = df['Loan_Status_Y']

# Convert boolean columns to int
X = X.apply(lambda col: col.astype(int) if col.dtype == 'bool' else col)

In [96]:

#Train-Test Split


X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

In [97]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [98]:
lr_model = LogisticRegression(
    C=0.5,                 # Slightly stronger regularization
    penalty='l2',
    solver='liblinear',
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)

# Train
lr_model.fit(X_train_scaled, y_train)

# Predictions
y_pred_lr = lr_model.predict(X_test_scaled)

# Train Accuracy
print("Train Accuracy:",
      lr_model.score(X_train_scaled, y_train)*100)

# Test Accuracy
print("Test Accuracy:",
      lr_model.score(X_test_scaled, y_test)*100)

# Classification Report
print("\nClassification Report (Logistic Regression):\n")
print(classification_report(y_test, y_pred_lr))

Train Accuracy: 74.82517482517483
Test Accuracy: 78.37837837837837

Classification Report (Logistic Regression):

              precision    recall  f1-score   support

           0       0.65      0.67      0.66        58
           1       0.85      0.83      0.84       127

    accuracy                           0.78       185
   macro avg       0.75      0.75      0.75       185
weighted avg       0.79      0.78      0.78       185



In [99]:
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)

dt_model.fit(X_train, y_train)

# Predictions
y_pred_dt = dt_model.predict(X_test)

# Accuracy
print("Train Accuracy:",
      dt_model.score(X_train, y_train)*100)

print("Test Accuracy:",
      dt_model.score(X_test, y_test)*100)

# Classification Report
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_dt))

Train Accuracy: 82.05128205128204
Test Accuracy: 82.70270270270271

Classification Report:

              precision    recall  f1-score   support

           0       0.81      0.59      0.68        58
           1       0.83      0.94      0.88       127

    accuracy                           0.83       185
   macro avg       0.82      0.76      0.78       185
weighted avg       0.83      0.83      0.82       185



In [100]:
# Create Model
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)

# Train Model
rf_model.fit(X_train, y_train)

# Predictions
y_pred_rf = rf_model.predict(X_test)

# Accuracy
print("Train Accuracy:",
      rf_model.score(X_train, y_train)*100)

print("Test Accuracy:",
      rf_model.score(X_test, y_test)*100)

# Classification Report
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_rf))

Train Accuracy: 82.75058275058275
Test Accuracy: 84.32432432432432

Classification Report:

              precision    recall  f1-score   support

           0       0.87      0.59      0.70        58
           1       0.84      0.96      0.89       127

    accuracy                           0.84       185
   macro avg       0.85      0.77      0.80       185
weighted avg       0.85      0.84      0.83       185



In [101]:
# SVM Model
svm_model = SVC(
    kernel='rbf',          # non-linear boundary
    C=1,                   # regularization
    gamma='scale',
    class_weight='balanced',
    probability=True,      # needed for ROC
    random_state=42
)
# Train
svm_model.fit(X_train_scaled, y_train)

# Accuracy
print("Train Accuracy:",
      svm_model.score(X_train_scaled, y_train)*100)

print("Test Accuracy:",
      svm_model.score(X_test_scaled, y_test)*100)

# Predictions
y_pred_svm = svm_model.predict(X_test_scaled)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_svm))


Train Accuracy: 85.3146853146853
Test Accuracy: 78.91891891891892

Classification Report:

              precision    recall  f1-score   support

           0       0.69      0.60      0.64        58
           1       0.83      0.87      0.85       127

    accuracy                           0.79       185
   macro avg       0.76      0.74      0.75       185
weighted avg       0.78      0.79      0.79       185



In [102]:

# xgb_model = XGBClassifier(
#     n_estimators=300,
#     learning_rate=0.05,
#     max_depth=4,
#     subsample=0.8,
#     colsample_bytree=0.8,
#     objective='binary:logistic',
#     eval_metric='logloss',
#     random_state=42
# )
xgb_model = XGBClassifier(
    n_estimators=100,        # reduce trees
    learning_rate=0.05,
    max_depth=2,             # shallower tree
    min_child_weight=3,      # prevent overfitting
    subsample=0.7,
    colsample_bytree=0.7,
    reg_alpha=1,             # L1 regularization
    reg_lambda=1,            # L2 regularization
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42
)


xgb_model.fit(X_train, y_train)

# Predictions
y_pred_xg = xgb_model.predict(X_test)

# Accuracy
print("Train Accuracy:",
      xgb_model.score(X_train, y_train)*100)

print("Test Accuracy:",
      xgb_model.score(X_test, y_test)*100)

# Classification Report
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_xg))

Train Accuracy: 80.65268065268066
Test Accuracy: 84.86486486486487

Classification Report:

              precision    recall  f1-score   support

           0       0.94      0.55      0.70        58
           1       0.83      0.98      0.90       127

    accuracy                           0.85       185
   macro avg       0.88      0.77      0.80       185
weighted avg       0.86      0.85      0.84       185



| Model               | Train Accuracy | Test Accuracy | Recall (Rejected – Class 0) | Verdict                 |
| ------------------- | -------------- | ------------- | --------------------------- | ----------------------- |
|  **XGBoost**      | 80.65%         | **84.86%**    | 0.55                        | Best Accuracy           |
| Random Forest    | 82.75%         | 84.32%        | 0.59                        | Most Stable             |
| Decision Tree    | 82.05%         | 82.70%        | 0.59                        | Most Interpretable      |
| Logistic Regression | 74.83%         | 78.38%        | **0.67**                    | Best for Risk Detection |
| SVM                 | 85.31%         | 78.91%        | 0.60                        | Mild Overfitting        |


After comparing Logistic Regression, SVM, Decision Tree, Random Forest, and XGBoost, the XGBoost classifier achieved the highest test accuracy of 84.86% with strong generalization and minimal overfitting. Therefore, XGBoost was selected as the final model for loan approval prediction.

In [104]:
import joblib

# Save model
joblib.dump(xgb_model, "loan_prediction_model.pkl")

print("Model saved successfully!")

Model saved successfully!
